In [1]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
loader= DirectoryLoader(
    path="./data_input",
    glob="**/*.*",
    loader_cls=UnstructuredFileLoader,
    show_progress=True,
    use_multithreading=True
)
docs = loader.load()

  0%|          | 0/5 [00:00<?, ?it/s]libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
 80%|████████  | 4/5 [00:13<00:02,  2.28s/it]

100%|██████████| 5/5 [00:20<00:00,  4.04s/it]


In [3]:
print(docs)
print(len(docs))

[Document(metadata={'source': 'data_input\\Part1-4_Minh_VN.md'}, page_content='Nghiên cứu và Thực hiện Trực quan hóa Dữ liệu bằng Python\n\n1. Mở đầu: Tầm quan trọng của trực quan hóa dữ liệu\n\nTrong phân tích dữ liệu hiện đại, trực quan hóa không đơn thuần là việc trình bày các con số dưới dạng hình ảnh, mà là một quy trình thiết yếu để chuyển hóa thông tin thô thành tri thức hữu dụng. Như Cole Nussbaumer Knaflic đã khẳng định trong tác phẩm Storytelling With Data: A Data Visualization Guide for Business Professionals:\n\n"Trực quan hóa dữ liệu là sự giao thoa giữa khoa học và nghệ thuật. Nó không chỉ đơn giản là tạo ra những biểu đồ đẹp mắt, mà là hiểu rõ bối cảnh, lựa chọn cách hiển thị phù hợp và loại bỏ những yếu tố gây nhiễu để tập trung vào thông điệp cốt lõi."\n\nViệc bỏ qua bước trực quan hóa và chỉ dựa vào các chỉ số thống kê tổng hợp có thể dẫn đến những sai lầm nghiêm trọng trong việc nhận diện bản chất của hiện tượng và đưa ra quyết định sai lệch.\n\n1.1. Phân tích thực n

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from pprint import pprint

MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]
# 1. SPLIT: Chia nhỏ tài liệu thành các đoạn (chunks)
# Giúp AI dễ dàng tìm kiếm và không bị quá giới hạn token của LLM
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Độ dài mỗi chunk (khoảng 1000 ký tự)
    chunk_overlap=100,  # Đoạn gối đầu giữa các chunk để không mất ngữ cảnh
    add_start_index=True,
    strip_whitespace=True,
    separators=MARKDOWN_SEPARATORS
)

chunks = text_splitter.split_documents(docs)
pprint(chunks)

[Document(metadata={'source': 'data_input\\Part1-4_Minh_VN.md', 'start_index': 0}, page_content='Nghiên cứu và Thực hiện Trực quan hóa Dữ liệu bằng Python\n\n1. Mở đầu: Tầm quan trọng của trực quan hóa dữ liệu\n\nTrong phân tích dữ liệu hiện đại, trực quan hóa không đơn thuần là việc trình bày các con số dưới dạng hình ảnh, mà là một quy trình thiết yếu để chuyển hóa thông tin thô thành tri thức hữu dụng. Như Cole Nussbaumer Knaflic đã khẳng định trong tác phẩm Storytelling With Data: A Data Visualization Guide for Business Professionals:\n\n"Trực quan hóa dữ liệu là sự giao thoa giữa khoa học và nghệ thuật. Nó không chỉ đơn giản là tạo ra những biểu đồ đẹp mắt, mà là hiểu rõ bối cảnh, lựa chọn cách hiển thị phù hợp và loại bỏ những yếu tố gây nhiễu để tập trung vào thông điệp cốt lõi."\n\nViệc bỏ qua bước trực quan hóa và chỉ dựa vào các chỉ số thống kê tổng hợp có thể dẫn đến những sai lầm nghiêm trọng trong việc nhận diện bản chất của hiện tượng và đưa ra quyết định sai lệch.\n\n1.1

In [7]:
import os
import shutil
from langchain_ollama import OllamaEmbeddings

# 1. Setup
db_directory = "./vector_db"
# Cấu hình Ollama Embeddings - Sử dụng model llama3 (8B)
# Đảm bảo bạn đã tải model bằng lệnh: ollama pull llama3
# LƯU Ý: Phải mở ứng dụng Ollama trên máy trước khi chạy code này
embeddings = OllamaEmbeddings(model="llama3")

# Xóa database cũ để tạo mới (tránh trùng lặp trong quá trình học tập)
if os.path.exists(db_directory):
    print("Đang xóa database cũ để tạo mới...")
    shutil.rmtree(db_directory)

# 2. Tạo Vector Database
print("Đang tạo Vector Database bằng Ollama llama3... (Quá trình này có thể mất vài phút)")
try:
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_directory
    )
    print(f"Thành công! Đã xử lý {len(docs)} tài liệu thành {len(chunks)} chunks.")
    print(f"Vector Database đã được lưu tại: {db_directory}")

    # 3. THỬ NGHIỆM TRUY XUẤT (Similarity Search)
    query = "Mục tiêu của khóa học English for Career Development là gì?"
    results = vector_db.similarity_search(query, k=2)

    print("\n--- Kết quả tìm kiếm thử nghiệm ---")
    for i, doc in enumerate(results):
        print(f"\nKết quả {i+1} (Nguồn: {doc.metadata.get('source')}):")
        print(f"Nội dung: {doc.page_content[:200]}...")
except Exception as e:
    print(f"\n[ERROR] Lỗi khi kết nối tới Ollama: {e}")
    print("\nLƯU Ý QUAN TRỌNG:")
    print("1. Đảm bảo ứng dụng Ollama đã được mở trên máy tính của bạn.")
    print("2. Nếu chưa pull model llama3, vui lòng chạy lệnh terminal: ollama pull llama3")

Đang xóa database cũ để tạo mới...


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: './vector_db\\chroma.sqlite3'